In [ ]:
class NodoAST:
    #Clase para todos los nodos del 'arbol ST
    pass #permite heredar de esta clase

    def traducirPy(self):
        #Traducci'on de C++ a Python
        raise NotImplementedError('Metodo traducirPy() no implementado en este Nodo.')
    
    #Traducci'on de C++ a Assembler
    def generarCodigo(self):
        raise NotImplementedError('Metodo generarCodigo() no implementado en este Nodo.') 
    
    def traducirJava(self):
        raise NotImplementedError('Metodo traducirJava() no implementado en este Nodo.')

class Nodoif(NodoAST):
    #Nodo que representa una sentencia if
    def __init__(self, condicion, cuerpo):
        self.condicion = condicion
        self.cuerpo_if = cuerpo_if
        self.cuerpo_else = cuerpo_else
    
    def traducirPy(self):
        cuerpo = '\n        '.join(inst.traducirPy() for inst in self.cuerpo_if)
        resultado = f"if {self.condicion.traducirPy()}:\n        {cuerpo}"
        if self.cuerpo_else:
            cuerpo_e = '\n        '.join(inst.traducirPy() for inst in self.cuerpo_else)
            resultado += f"\n    else:\n        {cuerpo_e}"
        return resultado

    def traducirJava(self):
        cuerpo = '\n        '.join(inst.traducirJava() for inst in self.cuerpo_if)
        resultado = f"if ({self.condicion.traducirJava()}) {{\n        {cuerpo}\n    }}"
        if self.cuerpo_else:
            cuerpo_e = '\n        '.join(inst.traducirJava() for inst in self.cuerpo_else)
            resultado += f" else {{\n        {cuerpo_e}\n    }}"
        return resultado

class NodoWhile(NodoAST):
    def __init__(self, condicion, cuerpo):
        self.condicion = condicion
        self.cuerpo = cuerpo

    def traducirPy(self):
        cuerpo_str = '\n        '.join(inst.traducirPy() for inst in self.cuerpo)
        return f"while {self.condicion.traducirPy()}:\n        {cuerpo_str}"

    def traducirJava(self):
        cuerpo_str = '\n        '.join(inst.traducirJava() for inst in self.cuerpo)
        return f"while ({self.condicion.traducirJava()}) {{\n        {cuerpo_str}\n    }}"

class NodoFor(NodoAST):
    def __init__(self, init, condicion, incremento, cuerpo):
        self.init = init           # NodoAsignacion
        self.condicion = condicion   # NodoOperacion/Expresion
        self.incremento = incremento # NodoAsignacion (ej: i = i + 1)
        self.cuerpo = cuerpo

    def traducirPy(self):
        # Python no tiene for(;;) tradicional, se traduce comúnmente a un while para mantener la lógica exacta
        init_py = self.init.traducirPy()
        cond_py = self.condicion.traducirPy()
        inc_py = self.incremento.traducirPy()
        cuerpo_str = '\n        '.join(inst.traducirPy() for inst in self.cuerpo)
        return f"{init_py}\n    while {cond_py}:\n        {cuerpo_str}\n        {inc_py}"

    def traducirJava(self):
        # En Java sí usamos la estructura nativa, pero quitamos el ';' extra de las asignaciones internas
        init_j = self.init.traducirJava().replace(';', '')
        cond_j = self.condicion.traducirJava()
        inc_j = self.incremento.traducirJava().replace(';', '')
        cuerpo_str = '\n        '.join(inst.traducirJava() for inst in self.cuerpo)
        return f"for ({init_j}; {cond_j}; {inc_j}) {{\n        {cuerpo_str}\n    }}"

class NodoPrograma(NodoAST):
    #Nodo que representa el programa completo
    def __init__(self, funciones, main):
        self.variables = []
        self.funciones = funciones
        self.main = main

class NodoLlamadaFuncion(NodoAST):
    #Nodo que representa una llamada a funci'on
    def __init__(self, nombre, argumentos):
        self.nombre_funcion = nombre
        self.argumentos = argumentos

class NodoFuncion(NodoAST):
    #Nodo que representa una funci'on
    def __init__(self, nombre, parametros, cuerpo):
        self.nombre = nombre
        self.parametros = parametros
        self.cuerpo = cuerpo
    
    #Traducci'on de C++ a Python
    def traducirPy(self):
        params = ', '.join(p.traducirPy() for p in self.parametros)
        cuerpo = '\n    '.join(c.traducirPy() for c in self.cuerpo)
        return f'def {self.nombre[1]}({params}):\n    {cuerpo}'
    
    #Traducci'on de C++ a Java
    def traducirJava(self):
        params = ', '.join(p.traducirJava() for p in self.parametros)
        cuerpo = '\n    '.join(c.traducirJava() for c in self.cuerpo)
        return f'public class Program {{\n    public static int {self.nombre[1]}({params}) {{\n        {cuerpo}\n    }}\n}}'

class NodoParametro(NodoAST):
    #Nodo que representa a un par'ametro de funci'on
    def __init__(self, tipo, nombre):
        self.tipo = tipo
        self.nombre = nombre
    
    def traducirPy(self):
        return self.nombre[1]
    
    def traducirJava(self):
        return f'{self.tipo[1]} {self.nombre[1]}'

class NodoAsignacion(NodoAST):
    #Nodo que representa una asignaci'on de variable
    def __init__(self, nombre, expresion):
        self.nombre = nombre
        self.expresion = expresion

    def traducirPy(self):
        return f'{self.nombre[1]} = {self.expresion.traducirPy()}'
    
    def traducirJava(self):
        return f'{self.nombre[1]} = {self.expresion.traducirJava()};'

class NodoOperacion(NodoAST):
    #Nodo que representa una operaci'on aritm'etica
    def __init__(self, left, operator, right):
        self.left = left
        self.operator = operator
        self.right = right

    def traducirPy(self):
        return f'{self.left.traducirPy()} {self.operator[1]} {self.right.traducirPy()}'
    
    def traducirJava(self):
        return f'({self.left.traducirJava()} {self.operator[1]} {self.right.traducirJava()})'

class NodoRetorno(NodoAST):
    #Nodo que representa la sentencia return
    def __init__(self, expresion):
        self.expresion = expresion

    def traducirPy(self):
        return f'return {self.expresion.traducirPy()}'
    
    def traducirJava(self):
        return f'return {self.expresion.traducirJava()};'

class NodoIdentificador(NodoAST):
    #Nodo que represnta a un identificador
    def __init__(self, nombre): #Por ahora se obvia el tipo porque no resulta necesario a menos que hayan constantes
        self.nombre = nombre

    def traducirPy(self):
        return self.nombre[1]
    
    def traducirJava(self):
        return self.nombre[1]

class NodoNumero(NodoAST):
    def __init__(self, valor):
        self.valor = valor

    def traducirPy(self):
        return str(self.valor[1])
    
    def traducirJava(self):
        return str(self.valor[1])
    
class NodoPrint(NodoAST):
    def __init__(self, expresion):
        self.expresion = expresion

    def traducirPy(self):
        return f'print({self.expresion.traducirPy()})'
    
    def traducirJava(self):
        return f'system.out.println({self.expresion.traducirJava()});'

In [ ]:
##    Analizador sintactico
class Parser:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0

    def obtener_token_actual(self):
        return self.tokens[self.pos] if self.pos < len(self.tokens) else None
    
    def coincidir(self, tipo_esperado):
        token_actual = self.obtener_token_actual()
        if token_actual and token_actual[0] == tipo_esperado: #Porque son duplas (tipo, valor [0])
            self.pos += 1
            return token_actual
        else:
            raise SyntaxError(f'Error sintactico: Se esperaba {tipo_esperado} pero se encontró: {token_actual}')
    
    def parsear(self):
        #Punto de entrada del analizador sintactico: Se espera un programa
        self.programa()

    def sentencia_if(self):
        self.coincidir('KEYWORD') # 'if'
        self.coincidir('DELIMITER') # '('
        condicion = self.expresion()
        self.coincidir('DELIMITER') # ')'
        self.coincidir('DELIMITER') # '{'
        cuerpo_if = self.cuerpo()
        self.coincidir('DELIMITER') # '}'
        
        cuerpo_else = None
        if self.obtener_token_actual() and self.obtener_token_actual()[1] == 'else':
            self.coincidir('KEYWORD') # 'else'
            self.coincidir('DELIMITER') # '{'
            cuerpo_else = self.cuerpo()
            self.coincidir('DELIMITER') # '}'
            
        return NodoIf(condicion, cuerpo_if, cuerpo_else)

    def sentencia_while(self):
        self.coincidir('KEYWORD') # 'while'
        self.coincidir('DELIMITER') # '('
        condicion = self.expresion()
        self.coincidir('DELIMITER') # ')'
        self.coincidir('DELIMITER') # '{'
        cuerpo = self.cuerpo()
        self.coincidir('DELIMITER') # '}'
        return NodoWhile(condicion, cuerpo)

    def sentencia_for(self):
        self.coincidir('KEYWORD') # 'for'
        self.coincidir('DELIMITER') # '('
        init = self.asignacion()     # i = 0; (ya consume el ;)
        condicion = self.expresion()
        self.coincidir('DELIMITER') # ';'
        
        # Para el incremento, necesitamos una asignación que no espere ';' al final
        nombre = self.coincidir('IDENTIFIER')
        self.coincidir('OPERATOR') # '='
        exp = self.expresion()
        incremento = NodoAsignacion(nombre, exp)
        
        self.coincidir('DELIMITER') # ')'
        self.coincidir('DELIMITER') # '{'
        cuerpo = self.cuerpo()
        self.coincidir('DELIMITER') # '}'
        return NodoFor(init, condicion, incremento, cuerpo)

    def programa(self):
        #Gramatica para un programa: funcion* main
        funciones = []
        while self.obtener_token_actual() and self.obtener_token_actual()[1] != 'main':
            funciones.append(self.funcion())
        main = self.funcion()  # Se espera la función main al final
        return NodoPrograma(funciones, main) # Se retorna el nodo programa con las funciones y el main  

    def funcion(self):
        #Gramatica para una funcion: int IDENTIFIER ( int IDENTIFIER ) { cuerpo }
        self.coincidir('KEYWORD') #tipo de retorno (Ej: int)
        nombre_funcion = self.coincidir('IDENTIFIER') # Nombre de la funcion
        self.coincidir('DELIMITER')  # Nombre de la funcion
        if nombre_funcion[1] == 'main':
            parametros = [] # main no tiene parametros
        else:
            parametros = self.parametros() # Regla para parametros
        self.coincidir('DELIMITER')  # Se espera un (
        self.coincidir('DELIMITER')  # Se espera un )
        cuerpo = self.cuerpo()  # Regla para el cuerpo de la funcion
        self.coincidir('DELIMITER')  # Se espera una {
        return NodoFuncion(nombre_funcion, parametros, cuerpo) # se retorna el nodo
    
    def parametros(self):
        lista_parametros = []
        # Reglas para parametros: int IDENTIFIER (, int IDENTIFIER)*
        tipo = self.coincidir('KEYWORD') #tipo de parametro
        nombre = self.coincidir('IDENTFIER') # Nombre del parametro
        lista_parametros.append(NodoParametro(nombre, tipo))
        while self.obtener_token_actual() and self.obtener_token_actual()[1] == ',':
            self.coincidir('DELIMITER') # Se espera una coma
            tipo = self.coincidir('KEYWORD')
            nombre = self.coincidir('IDENTIFIER')
            lista_parametros.append(NodoParametro(nombre, tipo))
        return lista_parametros
        
    def cuerpo(self):
        instrucciones = []
        # El cuerpo termina cuando encontramos un '}'
        while self.obtener_token_actual() and self.obtener_token_actual()[1] != '}':
            token = self.obtener_token_actual()
            valor = token[1]

            if valor == 'return':
                instrucciones.append(self.retorno())
            elif valor == 'print' or valor == 'println':
                instrucciones.append(self.sentencia_print())
            elif valor == 'if':
                instrucciones.append(self.sentencia_if())
            elif valor == 'while':
                instrucciones.append(self.sentencia_while())
            elif valor == 'for':
                instrucciones.append(self.sentencia_for())
            else:
                instrucciones.append(self.asignacion())
        return instrucciones

    def sentencia_print(self):
        token_tipo = self.coincidir('KEYWORD') # 'print' o 'println'
        es_ln = token_tipo[1] == 'println'
        
        self.coincidir('DELIMITER') # (
        expresion = self.expresion()
        self.coincidir('DELIMITER') # )
        self.coincidir('DELIMITER') # ;
        
        return NodoPrint(expresion, con_salto=es_ln)
    
    def sentencia_if(self):
        self.coincidir('KEYWORD')    # 'if'
        self.coincidir('DELIMITER')  # (
        condicion = self.expresion()
        self.coincidir('DELIMITER')  # )
        
        self.coincidir('DELIMITER')  # {
        bloque_if = self.cuerpo()
        self.coincidir('DELIMITER')  # }
        
        bloque_else = None
        token_actual = self.obtener_token_actual()
        if token_actual and token_actual[1] == 'else':
            self.coincidir('KEYWORD')   # 'else'
            self.coincidir('DELIMITER') # {
            bloque_else = self.cuerpo()
            self.coincidir('DELIMITER') # }
            
        return NodoIf(condicion, bloque_if, bloque_else)
    
    def sentencia_while(self):
        self.coincidir('KEYWORD')    # 'while'
        self.coincidir('DELIMITER')  # (
        condicion = self.expresion()
        self.coincidir('DELIMITER')  # )
        
        self.coincidir('DELIMITER')  # {
        cuerpo = self.cuerpo()
        self.coincidir('DELIMITER')  # }
        
        return NodoWhile(condicion, cuerpo)

    def sentencia_for(self):
        self.coincidir('KEYWORD')    # 'for'
        self.coincidir('DELIMITER')  # (
        
        # Asumiendo estructura: for(i = 0; i < 10; i = i + 1)
        inicializacion = self.asignacion() # Ya consume el ';'
        condicion = self.expresion()
        self.coincidir('DELIMITER')        # ;
        actualizacion = self.asignacion_sin_punto_coma() # Necesitarás este sub-método
        
        self.coincidir('DELIMITER')  # )
        self.coincidir('DELIMITER')  # {
        bloque = self.cuerpo()
        self.coincidir('DELIMITER')  # }
        
        return NodoFor(inicializacion, condicion, actualizacion, bloque)

    def asignacion(self):
        self.coincidir('KEYWORD')
        nombre = self.coincidir('IDENTIFIER')  # Se espera un identificador
        self.coincidir('OPERATOR')  # Se espera un operador de asignacion (=
        expresion = self.expresion()  # Se espera una expresion
        self.coincidir('DELIMITER')  # Se espera un ;
        return NodoAsignacion(nombre, expresion)

    def retorno(self):
        self.coincidir('KEYWORD')  # Se espera un return
        expresion = self.expresion()  # Se espera una expresion
        self.coincidir('DELIMITER')  # Se espera un ;
        return NodoRetorno(expresion)
    
    def expresion(self):
        izquierda = self.termino()
        while self.obtener_token_actual() and self.obtener_token_actual()[0] == 'OPERATOR':
            operador = self.coincidir('OPERATOR')
            derecha = self.termino()
            izquierda = NodoOperacion(izquierda, operador, derecha)
        return izquierda
    
    def termino(self):
        token_actual = self.obtener_token_actual()
        if token_actual[0] == 'IDENTIFIER':
            identificador = self.coincidir('IDENTIFIER')
            if self.obtener_token_actual() and self.obtener_token_actual()[1] == '(': # Si es una llamada a funcion
                self.coincidir('DELIMITER')  # Se espera un (
                argumentos = self.llamadaFuncion()  # Regla para argumentos
                self.coincidir('DELIMITER')  # Se espera un )
                return NodoLlamadaFuncion(identificador[1], argumentos)
            else:
                return NodoIdentificador(identificador)
            
        elif token_actual[0] == 'NUMBER':
            return NodoNumero(self.coincidir('NUMBER'))
        else:
            raise SyntaxError(f'Expresion no valida: {token_actual}')
        

    def llamadaFuncion(self):
        argumentos = []
        # Reglas para argumentos: identifier | number (, Identifier | Number)*
        sigue = True
        token = self.obtener_token_actual()
        while sigue:
            sigue = False
            if token[0] == 'NUMBER':
                argumento = NodoNumero(self.coincidir('NUMBER'))
            elif token[0] == 'IDENTIFIER':
                argumento = NodoIdentificador(self.coincidir('IDENTIFIER'))
            else:
                raise SyntaxError(f'Error de sintaxis, se esperaba un IDENTIFICADOR|NUMERO pero se encontro {token}')
            argumentos.append(argumento)
            if self.obtener_token_actual() and self.obtener_token_actual()[1] == ',':
                self.coincidir('DELIMITER') #Se espera una coma
                token = self.obtener_token_actual()
                sigue = True
        return argumentos